# Day 32 — Production RAG System

A clean, GitHub-ready version of the Day 32 notebook.

## Architecture

```text
PDF upload
   ↓
FastAPI `/upload-document`
   ↓
Text extraction → chunking → embeddings
   ↓
FAISS vector store (chunks + vectors + metadata)
   ↓
FastAPI `/ask`
   ↓
Semantic retrieval → LLM relevance grading
   ↓
Grounded context → LLM answer
```

### Demo vs production storage

This notebook intentionally uses **in-memory FAISS** and a Python dictionary so the RAG mechanics are easy to inspect. In an Azure production design, the original PDF could live in **Azure Blob Storage**, searchable chunks/vectors in **Azure AI Search**, and durable document/application metadata in a persistent database.

## 1. Install dependencies

In [ ]:
!pip install -q fastapi uvicorn python-multipart pypdf langchain langchain-community langchain-text-splitters langchain-huggingface langchain-groq faiss-cpu sentence-transformers reportlab

## 2. Imports and configuration
The Groq key is read from **Colab Secrets** (`GROQ_API_KEY`) and is never hard-coded in the notebook.

In [ ]:
import io
import os

from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.testclient import TestClient
from google.colab import userdata
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL = "llama-3.3-70b-versatile"  # Change here if your Groq account uses another supported model.

## 3. Create a controlled sample banking PDF
This keeps the demo reproducible: we know exactly what the source document says.

In [ ]:
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
from reportlab.lib.units import inch

SAMPLE_PDF = "sample_credit_risk_policy.pdf"

c = canvas.Canvas(SAMPLE_PDF, pagesize=A4)
pdf_text = c.beginText()
pdf_text.setTextOrigin(0.8 * inch, 11 * inch)
pdf_text.setLeading(18)

sample_text = """
Credit Risk Policy

High-risk corporate clients must undergo enhanced due diligence.
These clients must be reviewed every six months.

Medium-risk clients must be reviewed once every twelve months.

Low-risk clients should undergo a review every two years.

All customer risk assessments must be documented and stored
according to the bank's record-retention requirements.

If a client's risk profile changes significantly, an additional
review should be conducted regardless of the normal review schedule.

Compliance officers are responsible for ensuring that overdue
reviews are identified and escalated to the appropriate risk manager.

Customer identification information must remain accurate and
up to date throughout the banking relationship.

Any suspicious activity identified during a review must be reported
to the appropriate financial crime investigation team.
"""

for line in sample_text.strip().split("\n"):
    pdf_text.textLine(line)

c.drawText(pdf_text)
c.save()
print(f"Created: {SAMPLE_PDF}")

## 4. Document ingestion helpers
The PDF arrives as bytes, is wrapped as an in-memory file, converted to text, and then split into overlapping chunks.

In [ ]:
def extract_text_from_pdf(file_bytes):
    pdf_file = io.BytesIO(file_bytes)
    reader = PdfReader(pdf_file)

    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"

    return text.strip()


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)


def split_text_into_chunks(text):
    return text_splitter.split_text(text)

## 5. Embeddings, LLM, and relevance grader
Embeddings make semantic retrieval possible. The LLM grader is a second-stage check that rejects retrieved chunks that do not actually help answer the question.

In [ ]:
embedding_model = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
llm = ChatGroq(model=LLM_MODEL, temperature=0)


def grade_relevance(question, document_text):
    prompt = f"""
You are a relevance grader for a RAG system.

User question:
{question}

Retrieved document chunk:
{document_text}

Decide whether the chunk contains information that is useful for answering the question.
Respond with only one word: YES or NO.
"""
    response = llm.invoke(prompt)
    return response.content.strip().upper()

## 6. In-memory vector store and document registry
Each chunk is stored with `document_id` and `filename` metadata. This lets us trace retrieved knowledge back to its source and remove all chunks belonging to a deleted document.

In [ ]:
vector_store = None
documents = {}
next_document_id = 1


def store_chunks_in_vector_db(chunks, document_id, filename):
    global vector_store

    metadatas = [
        {"document_id": document_id, "filename": filename}
        for _ in chunks
    ]

    if vector_store is None:
        vector_store = FAISS.from_texts(
            texts=chunks,
            embedding=embedding_model,
            metadatas=metadatas,
        )
    else:
        vector_store.add_texts(texts=chunks, metadatas=metadatas)

## 7. FastAPI application
Endpoints: `GET /health`, `POST /upload-document`, `GET /documents`, `POST /ask`, and `DELETE /documents/{document_id}`.

In [ ]:
app = FastAPI(title="Production RAG API")


@app.get("/health")
def health_check():
    return {"status": "healthy", "service": "Production RAG API"}


@app.post("/upload-document")
async def upload_document(file: UploadFile = File(...)):
    global documents, next_document_id

    if not file.filename.lower().endswith(".pdf"):
        raise HTTPException(status_code=400, detail="Only PDF files are supported.")

    file_bytes = await file.read()
    if not file_bytes:
        raise HTTPException(status_code=400, detail="Uploaded file is empty.")

    try:
        text = extract_text_from_pdf(file_bytes)
    except Exception as exc:
        raise HTTPException(status_code=400, detail="Could not read the PDF.") from exc

    if not text.strip():
        raise HTTPException(status_code=400, detail="No extractable text found in the PDF.")

    chunks = split_text_into_chunks(text)
    document_id = str(next_document_id)
    next_document_id += 1

    store_chunks_in_vector_db(chunks, document_id, file.filename)

    documents[document_id] = {
        "filename": file.filename,
        "characters": len(text),
        "chunks": len(chunks),
    }

    return {
        "message": "Document uploaded successfully",
        "document_id": document_id,
        "filename": file.filename,
        "characters": len(text),
        "chunks": len(chunks),
    }


@app.get("/documents")
def list_documents():
    return {"count": len(documents), "documents": documents}


@app.post("/ask")
def ask_question(question: str):
    global vector_store

    question = question.strip()
    if not question:
        raise HTTPException(status_code=400, detail="Question cannot be empty.")

    if vector_store is None:
        raise HTTPException(status_code=400, detail="No documents have been uploaded.")

    retrieved_docs = vector_store.similarity_search(question, k=3)

    relevant_docs = []
    for doc in retrieved_docs:
        if grade_relevance(question, doc.page_content) == "YES":
            relevant_docs.append(doc)

    if not relevant_docs:
        return {
            "question": question,
            "answer": "I cannot answer this from the available documents.",
            "sources": [],
        }

    context = "\n\n".join(doc.page_content for doc in relevant_docs)
    prompt = f"""
You are a banking policy assistant.
Answer the user's question using ONLY the context below.
If the answer is not contained in the context, say:
"I cannot answer this from the available documents."

Context:
{context}

Question:
{question}

Answer:
"""

    response = llm.invoke(prompt)
    sources = list({doc.metadata.get("filename") for doc in relevant_docs})

    return {
        "question": question,
        "answer": response.content,
        "sources": sources,
    }


@app.delete("/documents/{document_id}")
def delete_document(document_id: str):
    global vector_store, documents

    if document_id not in documents:
        raise HTTPException(status_code=404, detail="Document not found.")

    remaining_texts = []
    remaining_metadatas = []

    if vector_store is not None:
        for doc in vector_store.docstore._dict.values():
            if doc.metadata.get("document_id") != document_id:
                remaining_texts.append(doc.page_content)
                remaining_metadatas.append(doc.metadata)

    deleted_document = documents.pop(document_id)

    if remaining_texts:
        vector_store = FAISS.from_texts(
            texts=remaining_texts,
            embedding=embedding_model,
            metadatas=remaining_metadatas,
        )
    else:
        vector_store = None

    return {
        "message": "Document deleted successfully",
        "document_id": document_id,
        "filename": deleted_document["filename"],
    }

## 8. End-to-end test
This test proves the full lifecycle: health → upload → list → ask → delete → list → ask after deletion.

In [ ]:
test_client = TestClient(app)

print("=== HEALTH ===")
health = test_client.get("/health")
print(health.status_code, health.json())

print("\n=== UPLOAD ===")
with open(SAMPLE_PDF, "rb") as file:
    upload = test_client.post(
        "/upload-document",
        files={"file": (SAMPLE_PDF, file, "application/pdf")},
    )
print(upload.status_code, upload.json())

document_id = upload.json()["document_id"]

print("\n=== DOCUMENTS ===")
listed = test_client.get("/documents")
print(listed.status_code, listed.json())

print("\n=== ASK ===")
asked = test_client.post(
    "/ask",
    params={"question": "How often are high-risk clients reviewed?"},
)
print(asked.status_code, asked.json())

print("\n=== DELETE ===")
deleted = test_client.delete(f"/documents/{document_id}")
print(deleted.status_code, deleted.json())

print("\n=== DOCUMENTS AFTER DELETE ===")
listed_after = test_client.get("/documents")
print(listed_after.status_code, listed_after.json())

print("\n=== ASK AFTER DELETE ===")
asked_after = test_client.post(
    "/ask",
    params={"question": "How often are high-risk clients reviewed?"},
)
print(asked_after.status_code, asked_after.json())

## 9. What this demonstrates

- PDF ingestion and text extraction
- Overlapping text chunking
- Hugging Face embeddings
- Semantic retrieval with FAISS
- Corrective-RAG-style LLM relevance grading
- Grounded answer generation
- FastAPI endpoints for health, upload, list, ask, and delete
- Source metadata attached to vector-store chunks
- Document lifecycle management
- Basic input/error validation

## Production evolution

This notebook is intentionally a learning/demo implementation. A production Azure deployment would typically replace temporary in-memory components with durable services:

```text
Client / UI
    ↓
FastAPI service
    ├── Azure Blob Storage → original PDFs
    ├── Azure AI Search    → persistent chunks + vectors + searchable metadata
    └── Persistent DB      → document/application metadata
```

Other production concerns include authentication/authorization, durable IDs, logging/observability, background ingestion jobs, rate limits, retries, monitoring, evaluation, and avoiding reliance on private FAISS internals such as `docstore._dict` for deletion.